# **Web Crawler**

This project implements a **basic web crawler** using Python.  
The crawler visits web pages, extracts useful information, and stores it in a SQLite database.

### Features:
- Crawl web pages starting from a seed URL
- Extract titles, text content, and links
- Store data in SQLite database
- Filter URLs by domain and file type
- Perform keyword search on crawled data

## Import Required Libraries

We import the necessary libraries for:
- HTTP requests → `requests`
- Logging → `logging`
- Database → `sqlite3`
- HTML parsing → `BeautifulSoup`
- URL handling → `urllib.parse`
- SSL warnings handling → `urllib3`

In [1]:
import requests
import logging
import sqlite3
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s:%(message)s')

## Crawler Class Design

The **Crawler** class manages the entire crawling process.

### Functionalities:
- Maintain visited and pending URLs
- Download web pages
- Extract content and links
- Store data in database
- Control crawl limits


## SQLite Database Setup

We create a SQLite database named `crawler.db`.

#### Table: crawled_pages
Stores:
- URL
- Page title
- Extracted text content
- Number of links on page
- Timestamp


In [2]:
class Crawler:

    SKIP_EXTENSIONS = ('.pdf', '.jpg', '.png', '.zip', '.docx', '.xlsx')

    def __init__(self, urls=None, allowed_domain=None, max_pages=50):
        self.visited_urls = []
        self.urls_to_visit = urls if urls is not None else []
        self.allowed_domain = allowed_domain
        self.max_pages = max_pages
        self.scraped_data = []

        # SQLite DB Setup
        self.conn = sqlite3.connect("crawler.db")
        self.cursor = self.conn.cursor()

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS crawled_pages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            url TEXT UNIQUE,
            title TEXT,
            content TEXT,
            links_count INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)
        self.conn.commit()

    def download_url(self, url):
        return requests.get(url, verify=False, timeout=10).text

    def is_valid_url(self, url):
        if not url:
            return False

        parsed = urlparse(url)

        if parsed.scheme not in ('http', 'https'):
            return False

        if self.allowed_domain and self.allowed_domain not in parsed.netloc:
            return False

        if parsed.path.lower().endswith(self.SKIP_EXTENSIONS):
            return False

        return True

    def get_linked_urls(self, url, html):
        soup = BeautifulSoup(html, 'html.parser')

        for link in soup.find_all('a'):
            path = link.get('href')

            if path:
                if path.startswith('/'):
                    path = urljoin(url, path)

                if self.is_valid_url(path):
                    yield path

    def extract_content(self, url, html):
        soup = BeautifulSoup(html, 'html.parser')

        for tag in soup(['script', 'style', 'nav', 'footer']):
            tag.decompose()

        return {
            'url': url,
            'title': soup.title.string.strip() if soup.title else 'No title',
            'text': soup.get_text(separator=' ', strip=True)[:3000],
            'links_count': len(soup.find_all('a'))
        }

    def save_to_db(self, page):
        try:
            self.cursor.execute("""
                INSERT OR IGNORE INTO crawled_pages (url, title, content, links_count)
                VALUES (?, ?, ?, ?)
            """, (
                page['url'],
                page['title'],
                page['text'],
                page['links_count']
            ))
            self.conn.commit()
        except Exception as e:
            logging.error(f"DB Insert Error: {e}")

    def add_url_to_visit(self, url):
        if url not in self.visited_urls and url not in self.urls_to_visit:
            self.urls_to_visit.append(url)

    def crawl(self, url):
        html = self.download_url(url)
        page_data = self.extract_content(url, html)

        self.scraped_data.append(page_data)

        # Save to SQLite
        self.save_to_db(page_data)

        for linked_url in self.get_linked_urls(url, html):
            self.add_url_to_visit(linked_url)

    def run(self):
        while self.urls_to_visit:
            if len(self.visited_urls) >= self.max_pages:
                logging.info(f'Reached max page limit of {self.max_pages}. Stopping.')
                break

            url = self.urls_to_visit.pop(0)

            if url in self.visited_urls:
                continue

            logging.info(f'Crawling: {url}')

            try:
                self.crawl(url)
            except Exception:
                logging.exception(f'Failed to crawl: {url}')
            finally:
                self.visited_urls.append(url)

        logging.info(f'Done. Crawled {len(self.visited_urls)} pages.')

In [3]:
crawler = Crawler(
    urls=['https://www.w3schools.com/'],
    allowed_domain='w3schools.com',
    max_pages=20
)

crawler.run()

## View Crawled Data

We fetch:
- URL
- Title

In [4]:
conn = sqlite3.connect("crawler.db")
cursor = conn.cursor()

cursor.execute("SELECT url, title FROM crawled_pages LIMIT 10")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('https://www.w3schools.com/', 'W3Schools Online Web Tutorials')
('https://www.w3schools.com', 'W3Schools Online Web Tutorials')
('https://profile.w3schools.com/log-in', 'W3Schools')
('https://campus.w3schools.com/collections/course-catalog', 'Certification Course Catalog — W3Schools.com')
('https://order.w3schools.com/plans', 'Become a Plus User at W3schools - Upgrade to Improve Your Learning Experience')
('https://www.w3schools.com/academy/index.php', "W3Schools Academy - Train Your Team with the World's Largest Web Developer Site")
('https://www.w3schools.com/spaces/index.php', 'Create a Website | Website Builder | W3Schools.com | W3Schools Spaces')
('https://www.w3schools.com/bootcamp/index.php', 'W3Schools Bootcamps - Launch Your Tech Career')
('https://spaces.w3schools.com', 'Spaces - W3schools')
('https://pathfinder.w3schools.com', 'W3Schools Pathfinder')


## Detailed Data View

Displays:
- ID
- URL
- Title
- Links count
- Timestamp
- Text preview

In [5]:
import sqlite3

conn = sqlite3.connect("crawler.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM crawled_pages")
rows = cursor.fetchall()

for row in rows:
    print("-" * 80)
    print(f"ID          : {row[0]}")
    print(f"URL         : {row[1]}")
    print(f"TITLE       : {row[2]}")
    print(f"LINKS COUNT : {row[4]}")
    print(f"CREATED AT  : {row[5]}")
    print(f"TEXT (preview): {row[3][:200]}...")  # show only first 200 chars

conn.close()

--------------------------------------------------------------------------------
ID          : 1
URL         : https://www.w3schools.com/
TITLE       : W3Schools Online Web Tutorials
LINKS COUNT : 225
CREATED AT  : 2026-03-29 14:05:00
TEXT (preview): W3Schools Online Web Tutorials Menu Search field × See More Sign In ★ +1 Get Certified Upgrade Teachers Spaces Bootcamps Get Certified Upgrade Teachers Spaces Bootcamps ❮ ❯ HTML CSS JAVASCRIPT SQL PYT...
--------------------------------------------------------------------------------
ID          : 2
URL         : https://www.w3schools.com
TITLE       : W3Schools Online Web Tutorials
LINKS COUNT : 225
CREATED AT  : 2026-03-29 14:05:01
TEXT (preview): W3Schools Online Web Tutorials Menu Search field × See More Sign In ★ +1 Get Certified Upgrade Teachers Spaces Bootcamps Get Certified Upgrade Teachers Spaces Bootcamps ❮ ❯ HTML CSS JAVASCRIPT SQL PYT...
--------------------------------------------------------------------------------
ID        

## Search in Crawled Data

We perform a keyword search using SQL:

- Uses `LIKE` operator
- Searches inside page content
- Returns matching URLs and titles

Example:
Search for "HTML" across all crawled pages.

In [6]:
query = "HTML"

conn = sqlite3.connect("crawler.db")
cursor = conn.cursor()

cursor.execute("""
SELECT url, title
FROM crawled_pages
WHERE content LIKE ?
LIMIT 5
""", ('%' + query + '%',))

results = cursor.fetchall()

for r in results:
    print(r)

conn.close()

('https://www.w3schools.com/', 'W3Schools Online Web Tutorials')
('https://www.w3schools.com', 'W3Schools Online Web Tutorials')
('https://campus.w3schools.com/collections/course-catalog', 'Certification Course Catalog — W3Schools.com')
('https://order.w3schools.com/plans', 'Become a Plus User at W3schools - Upgrade to Improve Your Learning Experience')
('https://www.w3schools.com/academy/index.php', "W3Schools Academy - Train Your Team with the World's Largest Web Developer Site")
